In [1]:
import os
import sys

# --- Windows DLL Search Path Fix ---
site_pkgs = os.path.join(os.path.dirname(sys.executable), "Lib", "site-packages")
torch_lib = os.path.join(site_pkgs, "torch", "lib")
if os.path.exists(torch_lib):
  if hasattr(os, "add_dll_directory"):
    os.add_dll_directory(torch_lib)
  os.environ["PATH"] += os.pathsep + torch_lib

import glob

for bin_path in glob.glob(os.path.join(site_pkgs, "nvidia", "*", "bin")):
  if os.path.exists(bin_path):
    if hasattr(os, "add_dll_directory"):
      os.add_dll_directory(bin_path)
    os.environ["PATH"] += os.pathsep + bin_path

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
# -----------------------------------

from pathlib import Path
import torch
import torchaudio
from df.enhance import enhance, init_df
from pydub import AudioSegment

# --- Configuration ---
INPUT_DIR = Path("audio")
OUTPUT_DIR = Path("audio_ai_clean")
AUDIO_EXTENSIONS = {".amr", ".mp3", ".wav", ".m4a", ".flac", ".ogg"}


def main():
  if not INPUT_DIR.exists():
    print(f"Error: Directory '{INPUT_DIR}' not found.")
    return

  OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

  audio_files = [
      f for f in INPUT_DIR.rglob("*") if f.suffix.lower() in AUDIO_EXTENSIONS
  ]
  if not audio_files:
    print(f"No audio files found in '{INPUT_DIR}'.")
    return

  print("Initializing DeepFilterNet AI speech enhancement model on GPU...")
  model, df_state, _ = init_df()
  if torch.cuda.is_available():
    model.to("cuda")

  for audio_path in audio_files:
    print(f"\nProcessing: {audio_path.name}")
    temp_wav_path = None
    try:
      # 1. Pre-convert AMR / compressed formats to standard WAV using pydub
      temp_wav_path = audio_path.with_suffix(".tmp.wav")
      sound = AudioSegment.from_file(str(audio_path))
      sound.export(str(temp_wav_path), format="wav")

      # 2. Load the converted WAV file as a CPU tensor
      audio, sample_rate = torchaudio.load(str(temp_wav_path))

      # 3. Clean up temporary file immediately
      if temp_wav_path.exists():
        temp_wav_path.unlink()

      # 4. Convert to mono if multi-channel
      if audio.shape[0] > 1:
        audio = audio.mean(dim=0, keepdim=True)

      # 5. Resample to DeepFilterNet native rate (48kHz)
      if sample_rate != df_state.sr():
        resampler = torchaudio.transforms.Resample(
            orig_freq=sample_rate, new_freq=df_state.sr()
        )
        audio = resampler(audio)

      # NOTE: Keep audio on CPU here. DeepFilterNet handles model-side GPU compute.
      enhanced = enhance(model, df_state, audio)

      # Ensure output tensor is on CPU before saving
      if enhanced.is_cuda:
        enhanced = enhanced.cpu()

      output_path = (
          OUTPUT_DIR / audio_path.relative_to(INPUT_DIR)
      ).with_suffix(".wav")
      output_path.parent.mkdir(parents=True, exist_ok=True)

      torchaudio.save(str(output_path), enhanced, df_state.sr())
      print(f"Successfully cleaned and saved: {output_path.name}")

    except Exception as e:
      print(f"Error processing {audio_path.name}: {e}")
      if temp_wav_path and temp_wav_path.exists():
        temp_wav_path.unlink()

  print(
      "\nAI audio enhancement complete! Clean files are saved in:"
      f" '{OUTPUT_DIR}/'"
  )


if __name__ == "__main__":
  main()

Initializing DeepFilterNet AI speech enhancement model on GPU...
2026-07-23 12:35:45 | INFO     | DF | Running on torch 2.6.0+cu124
2026-07-23 12:35:45 | INFO     | DF | Running on host HP-G9
2026-07-23 12:35:45 | INFO     | DF | Loading model settings of DeepFilterNet3
2026-07-23 12:35:45 | INFO     | DF | Using DeepFilterNet3 model at C:\Users\micha\AppData\Local\DeepFilterNet\DeepFilterNet\Cache\DeepFilterNet3
2026-07-23 12:35:45 | INFO     | DF | Initializing model `deepfilternet3`


c:\Users\micha\anaconda3\envs\ai-audio-env\lib\site-packages\df\io.py:9: UserWarning: `torchaudio.backend.common.AudioMetaData` has been moved to `torchaudio.AudioMetaData`. Please update the import path.
  from torchaudio.backend.common import AudioMetaData


2026-07-23 12:35:45 | INFO     | DF | Found checkpoint C:\Users\micha\AppData\Local\DeepFilterNet\DeepFilterNet\Cache\DeepFilterNet3\checkpoints\model_120.ckpt.best with epoch 120
2026-07-23 12:35:45 | INFO     | DF | Running on device cuda:0
2026-07-23 12:35:45 | INFO     | DF | Model loaded

Processing: phone_20260628-000120_9126526920.amr
Successfully cleaned and saved: phone_20260628-000120_9126526920.wav

Processing: phone_20260723-093140_9126526988.amr
Successfully cleaned and saved: phone_20260723-093140_9126526988.wav

AI audio enhancement complete! Clean files are saved in: 'audio_ai_clean/'


In [3]:
import os
import sys

# --- Windows DLL Search Path Fix ---
site_pkgs = os.path.join(os.path.dirname(sys.executable), "Lib", "site-packages")
torch_lib = os.path.join(site_pkgs, "torch", "lib")
if os.path.exists(torch_lib):
  if hasattr(os, "add_dll_directory"):
    os.add_dll_directory(torch_lib)
  os.environ["PATH"] += os.pathsep + torch_lib

import glob

for bin_path in glob.glob(os.path.join(site_pkgs, "nvidia", "*", "bin")):
  if os.path.exists(bin_path):
    if hasattr(os, "add_dll_directory"):
      os.add_dll_directory(bin_path)
    os.environ["PATH"] += os.pathsep + bin_path

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
# -----------------------------------

from pathlib import Path
import torch
import torchaudio
from df.enhance import enhance, init_df
from pydub import AudioSegment

# --- Configuration ---
INPUT_DIR = Path("audio")
OUTPUT_DIR = Path("audio_ai_clean")
AUDIO_EXTENSIONS = {".amr", ".mp3", ".wav", ".m4a", ".flac", ".ogg"}


def main():
  if not INPUT_DIR.exists():
    print(f"Error: Directory '{INPUT_DIR}' not found.")
    return

  OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

  audio_files = [
      f for f in INPUT_DIR.rglob("*") if f.suffix.lower() in AUDIO_EXTENSIONS
  ]
  if not audio_files:
    print(f"No audio files found in '{INPUT_DIR}'.")
    return

  print("Initializing DeepFilterNet AI speech enhancement model on GPU...")
  model, df_state, _ = init_df()
  if torch.cuda.is_available():
    model.to("cuda")

  for audio_path in audio_files:
    print(f"\nProcessing Dual-Channel Call: {audio_path.name}")
    temp_wav_path = None
    try:
      # 1. Convert AMR file to WAV using pydub
      temp_wav_path = audio_path.with_suffix(".tmp.wav")
      sound = AudioSegment.from_file(str(audio_path))
      sound.export(str(temp_wav_path), format="wav")

      # 2. Load audio tensor
      audio, sample_rate = torchaudio.load(str(temp_wav_path))

      if temp_wav_path.exists():
        temp_wav_path.unlink()

      # 3. Resample to DeepFilterNet native rate (48kHz) if needed
      if sample_rate != df_state.sr():
        resampler = torchaudio.transforms.Resample(
            orig_freq=sample_rate, new_freq=df_state.sr()
        )
        audio = resampler(audio)

      # 4. Handle Dual Channels (Stereo Phone Recording) vs Mono
      if audio.shape[0] >= 2:
        print(" -> Detected dual-channel call recording. Processing channels separately...")
        ch1 = audio[0:1, :]  # Local speaker (you)
        ch2 = audio[1:2, :]  # Remote speaker (operator)

        # Apply an extra +12 dB boost specifically to the quieter remote channel
        ch2 = ch2 * 4.0  

        # Run DeepFilterNet AI enhancement on each channel independently
        enhanced_ch1 = enhance(model, df_state, ch1)
        enhanced_ch2 = enhance(model, df_state, ch2)

        # Recombine both enhanced channels back into a stereo file
        enhanced = torch.cat([enhanced_ch1, enhanced_ch2], dim=0)
      else:
        print(" -> Detected mono recording. Applying standard enhancement...")
        enhanced = enhance(model, df_state, audio)

      if enhanced.is_cuda:
        enhanced = enhanced.cpu()

      output_path = (
          OUTPUT_DIR / audio_path.relative_to(INPUT_DIR)
      ).with_suffix(".wav")
      output_path.parent.mkdir(parents=True, exist_ok=True)

      torchaudio.save(str(output_path), enhanced, df_state.sr())
      print(f"Successfully balanced and saved: {output_path.name}")

    except Exception as e:
      print(f"Error processing {audio_path.name}: {e}")
      if temp_wav_path and temp_wav_path.exists():
        temp_wav_path.unlink()

  print(
      "\nCall enhancement complete! Check your balanced files in:"
      f" '{OUTPUT_DIR}/'"
  )


if __name__ == "__main__":
  main()

Initializing DeepFilterNet AI speech enhancement model on GPU...
2026-07-23 12:38:38 | INFO     | DF | Loading model settings of DeepFilterNet3
2026-07-23 12:38:38 | INFO     | DF | Using DeepFilterNet3 model at C:\Users\micha\AppData\Local\DeepFilterNet\DeepFilterNet\Cache\DeepFilterNet3
2026-07-23 12:38:38 | INFO     | DF | Initializing model `deepfilternet3`
2026-07-23 12:38:38 | INFO     | DF | Found checkpoint C:\Users\micha\AppData\Local\DeepFilterNet\DeepFilterNet\Cache\DeepFilterNet3\checkpoints\model_120.ckpt.best with epoch 120
2026-07-23 12:38:38 | INFO     | DF | Running on device cuda:0
2026-07-23 12:38:38 | INFO     | DF | Model loaded

Processing Dual-Channel Call: phone_20260628-000120_9126526920.amr
 -> Detected mono recording. Applying standard enhancement...
Successfully balanced and saved: phone_20260628-000120_9126526920.wav

Processing Dual-Channel Call: phone_20260723-093140_9126526988.amr
 -> Detected mono recording. Applying standard enhancement...
Successfully